# 📄 Document Image Enhancement: Deblurring & Denoising
## Optimized for Google Colab Free Tier (T4 GPU · ~300 paired images)

---

### ✅ Setup Instructions

1. **Upload your dataset to Google Drive** at `MyDrive/document_dataset/`
   - `degraded/` folder — your blurry/noisy document images
   - `clean/` folder — matching clean ground-truth images
   - **File names MUST match** between the two folders (e.g. `doc001.png` in both)
2. In the Colab menu: **Runtime → Change runtime type → Hardware accelerator → GPU (T4)**
3. **Run all cells in order** (Runtime → Run all)

---

### 📌 What this notebook does

| Section | Description | Time |
|---------|-------------|------|
| Classical Enhancement | CLAHE, denoising, sharpening — **instant results, no training** | < 1 min |
| Deep Learning (U-Net) | Patch-based training on ~54 K patches from 300 images | ~30–45 min on T4 |
| Results & Comparison | PSNR / SSIM metrics, side-by-side visuals, bar chart | < 1 min |
| Single Image Testing | Enhance any image with the trained model | < 5 s |

---

### ⚠️ If Colab Disconnects
- The best model is **automatically saved to Google Drive** after every improving epoch.
- Re-run all cells — training will **resume from the last checkpoint** automatically.

### ⚠️ If No GPU Is Available
- Classical methods still work perfectly.
- Deep-learning training will be much slower (~10× longer); reduce `EPOCHS` to 30–50.


## Cell 1 · Environment Setup

In [ ]:
# ── Mount Google Drive ────────────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted")
except Exception as e:
    print(f"⚠️  Could not mount Google Drive: {e}")
    print("💡 Fix: Make sure you are running in Google Colab and allow the permissions pop-up.")


In [ ]:
# ── GPU / CPU Detection ───────────────────────────────────────────────────────
import subprocess, os, gc

try:
    gpu_info = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total',
                                        '--format=csv,noheader'], encoding='utf-8').strip()
    print(f"✅ GPU detected: {gpu_info}")
    HAS_GPU = True
except Exception:
    print("⚠️  No GPU detected — training will run on CPU (much slower).")
    print("💡 Fix: Runtime → Change runtime type → GPU (T4)")
    HAS_GPU = False

# System RAM
try:
    import psutil
    ram_gb = psutil.virtual_memory().total / 1e9
    print(f"💾 System RAM: {ram_gb:.1f} GB")
except Exception:
    print("💾 System RAM: (psutil not available)")


In [ ]:
# ── Install / Import Dependencies ─────────────────────────────────────────────
try:
    import tensorflow as tf
    import numpy as np
    import cv2
    import matplotlib.pyplot as plt
    import matplotlib.gridspec as gridspec
    from pathlib import Path
    import random, shutil, csv, time

    print(f"✅ TensorFlow {tf.__version__}")
    print(f"✅ NumPy {np.__version__}")
    print(f"✅ OpenCV {cv2.__version__}")
    print("✅ All imports successful")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("💡 Fix: Run  !pip install tensorflow opencv-python-headless  then restart runtime")


In [ ]:
# ── Reproducibility Seeds ─────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
print(f"✅ Random seeds set to {SEED}")


In [ ]:
# ── Mixed Precision (halves VRAM usage on T4) ─────────────────────────────────
try:
    if HAS_GPU:
        tf.keras.mixed_precision.set_global_policy('mixed_float16')
        print("✅ Mixed precision enabled (float16 — saves ~50%% VRAM)")
    else:
        tf.keras.mixed_precision.set_global_policy('float32')
        print("ℹ️  CPU mode: using float32")
except Exception as e:
    print(f"⚠️  Mixed precision not set: {e}")

gc.collect()
print("✅ Environment ready")


## Cell 2 · Configuration

> **Edit the values in this cell only.** All other cells read from these variables.

In [ ]:
# ===== USER CONFIGURATION =====
DATASET_PATH   = '/content/drive/MyDrive/document_dataset/'
CHECKPOINT_DIR = '/content/drive/MyDrive/document_enhancement_checkpoints/'
RESULTS_DIR    = '/content/drive/MyDrive/document_enhancement_results/'

PATCH_SIZE     = 128    # 128×128 patches (saves 4× memory vs 256×256)
PATCH_STRIDE   = 64     # 50%% overlap → ~180 patches per image
BATCH_SIZE     = 8      # Safe for T4 with mixed precision; try 16 if no OOM errors
EPOCHS         = 150    # EarlyStopping will stop sooner if converged
LEARNING_RATE  = 1e-3   # Start aggressive; ReduceLROnPlateau decays automatically
VAL_SPLIT      = 0.2    # 80%% train / 20%% validation  (split at IMAGE level)
IMG_CHANNELS   = 1      # Grayscale (documents) — saves 3× memory vs RGB
# ===============================

# Derived paths
CHECKPOINT_PATH   = CHECKPOINT_DIR + 'best_model.keras'
DEGRADED_DIR      = DATASET_PATH + 'degraded/'
CLEAN_DIR         = DATASET_PATH + 'clean/'

# Create output directories
for d in [CHECKPOINT_DIR, RESULTS_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

print("✅ Configuration loaded")
print(f"   Dataset  : {DATASET_PATH}")
print(f"   Patches  : {PATCH_SIZE}×{PATCH_SIZE}, stride={PATCH_STRIDE}")
print(f"   Batch    : {BATCH_SIZE},  Epochs: {EPOCHS},  LR: {LEARNING_RATE}")


## Cell 3 · Dataset Loading & Validation

In [ ]:
try:
    degraded_dir = Path(DEGRADED_DIR)
    clean_dir    = Path(CLEAN_DIR)

    if not degraded_dir.exists():
        raise FileNotFoundError(f"Degraded folder not found: {DEGRADED_DIR}")
    if not clean_dir.exists():
        raise FileNotFoundError(f"Clean folder not found: {CLEAN_DIR}")

    IMG_EXTENSIONS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
    degraded_files = sorted([f for f in degraded_dir.iterdir()
                              if f.suffix.lower() in IMG_EXTENSIONS])
    clean_files    = sorted([f for f in clean_dir.iterdir()
                              if f.suffix.lower() in IMG_EXTENSIONS])

    degraded_names = {f.name for f in degraded_files}
    clean_names    = {f.name for f in clean_files}
    matched_names  = sorted(degraded_names & clean_names)

    unmatched_deg  = degraded_names - clean_names
    unmatched_cln  = clean_names    - degraded_names

    if unmatched_deg:
        print(f"⚠️  {len(unmatched_deg)} degraded files have no matching clean file.")
    if unmatched_cln:
        print(f"⚠️  {len(unmatched_cln)} clean files have no matching degraded file.")
    if len(matched_names) == 0:
        raise ValueError("No matched pairs found! Check that file names match in both folders.")

    PAIRED_DEGRADED = [degraded_dir / n for n in matched_names]
    PAIRED_CLEAN    = [clean_dir    / n for n in matched_names]

    print(f"✅ Found {len(matched_names)} paired images")

except FileNotFoundError as e:
    print(f"❌ Error: {e}")
    print(f"💡 Fix: Upload your dataset to Google Drive at {DATASET_PATH}")
    print("       with sub-folders  degraded/  and  clean/")
    raise
except Exception as e:
    print(f"❌ Unexpected error: {e}")
    raise


In [ ]:
# ── Preview 6 sample pairs ────────────────────────────────────────────────────
try:
    n_preview = min(6, len(PAIRED_DEGRADED))
    fig, axes = plt.subplots(2, n_preview, figsize=(3 * n_preview, 6))

    for i in range(n_preview):
        deg = cv2.imread(str(PAIRED_DEGRADED[i]), cv2.IMREAD_GRAYSCALE)
        cln = cv2.imread(str(PAIRED_CLEAN[i]),    cv2.IMREAD_GRAYSCALE)
        axes[0, i].imshow(deg, cmap='gray'); axes[0, i].set_title(f"Degraded {i+1}", fontsize=8)
        axes[1, i].imshow(cln, cmap='gray'); axes[1, i].set_title(f"Clean {i+1}",    fontsize=8)
        axes[0, i].axis('off'); axes[1, i].axis('off')

    plt.suptitle("Sample Pairs (top: degraded, bottom: clean)", fontsize=12)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"❌ Preview error: {e}")
    print("💡 Fix: Make sure your images are valid PNG/JPG files")


## Cell 4 · Patch Extraction

> **Why patches?** Instead of training on ~300 full images, we extract overlapping
> 128×128 patches (stride=64 → 50% overlap). A 1000×800 image yields ~180 patches,
> giving us **~54,000 training patches** — a 180× data multiplication!
>
> The train/validation split is done at **image level** (before patch extraction)
> to prevent data leakage.

*Expected runtime: 1–5 minutes depending on image sizes.*


In [ ]:
import numpy as np
from pathlib import Path
import cv2, gc

def extract_patches(img, patch_size, stride):
    """Return list of (patch_size x patch_size) numpy patches from a 2-D image.
    Images smaller than patch_size are padded before extraction.
    """
    h, w = img.shape[:2]
    # Pad image if smaller than patch_size so at least one patch is produced
    if h < patch_size or w < patch_size:
        pad_h = max(patch_size, h)
        pad_w = max(patch_size, w)
        padded = np.zeros((pad_h, pad_w), dtype=img.dtype)
        padded[:h, :w] = img
        img, h, w = padded, pad_h, pad_w
    patches = []
    for y in range(0, h - patch_size + 1, stride):
        for x in range(0, w - patch_size + 1, stride):
            patches.append(img[y:y+patch_size, x:x+patch_size])
    return patches

try:
    # ── Image-level split (prevents data leakage) ──────────────────────────
    indices = list(range(len(PAIRED_DEGRADED)))
    random.shuffle(indices)
    split = int(len(indices) * (1 - VAL_SPLIT))
    train_indices = indices[:split]
    val_indices   = indices[split:]

    TRAIN_DEG = [PAIRED_DEGRADED[i] for i in train_indices]
    TRAIN_CLN = [PAIRED_CLEAN[i]    for i in train_indices]
    VAL_DEG   = [PAIRED_DEGRADED[i] for i in val_indices]
    VAL_CLN   = [PAIRED_CLEAN[i]    for i in val_indices]

    print(f"Image split → Train: {len(TRAIN_DEG)}, Val: {len(VAL_DEG)}")

    # ── Extract & save patches ─────────────────────────────────────────────
    PATCH_DIR = Path('/content/patches')
    for sub in ['train/degraded', 'train/clean', 'val/degraded', 'val/clean']:
        (PATCH_DIR / sub).mkdir(parents=True, exist_ok=True)

    def save_patches(img_paths_deg, img_paths_cln, split_name):
        count = 0
        for deg_path, cln_path in zip(img_paths_deg, img_paths_cln):
            deg = cv2.imread(str(deg_path), cv2.IMREAD_GRAYSCALE)
            cln = cv2.imread(str(cln_path), cv2.IMREAD_GRAYSCALE)
            if deg is None or cln is None:
                print(f"⚠️  Could not read {deg_path.name} — skipping")
                continue
            # Resize clean to match degraded if needed
            if deg.shape != cln.shape:
                cln = cv2.resize(cln, (deg.shape[1], deg.shape[0]))
            deg_patches = extract_patches(deg, PATCH_SIZE, PATCH_STRIDE)
            cln_patches = extract_patches(cln, PATCH_SIZE, PATCH_STRIDE)
            for i, (dp, cp) in enumerate(zip(deg_patches, cln_patches)):
                stem = deg_path.stem
                cv2.imwrite(str(PATCH_DIR / split_name / 'degraded' / f'{stem}_{i:04d}.png'), dp)
                cv2.imwrite(str(PATCH_DIR / split_name / 'clean'    / f'{stem}_{i:04d}.png'), cp)
                count += 1
        return count

    train_count = save_patches(TRAIN_DEG, TRAIN_CLN, 'train')
    val_count   = save_patches(VAL_DEG,   VAL_CLN,   'val')

    TRAIN_DEG_PATCHES = sorted((PATCH_DIR / 'train/degraded').glob('*.png'))
    TRAIN_CLN_PATCHES = sorted((PATCH_DIR / 'train/clean').glob('*.png'))
    VAL_DEG_PATCHES   = sorted((PATCH_DIR / 'val/degraded').glob('*.png'))
    VAL_CLN_PATCHES   = sorted((PATCH_DIR / 'val/clean').glob('*.png'))

    print(f"\n✅ Patch extraction complete")
    print(f"   Training patches  : {train_count:,}")
    print(f"   Validation patches: {val_count:,}")
    print(f"   Total patches     : {train_count + val_count:,}")
    print(f"   Data multiplication: ~{(train_count + val_count) / len(PAIRED_DEGRADED):.0f}× original images")

    gc.collect()

except Exception as e:
    print(f"❌ Patch extraction failed: {e}")
    print("💡 Fix: Make sure images in degraded/ and clean/ are valid and readable")
    raise


In [ ]:
# ── Preview sample patches ────────────────────────────────────────────────────
try:
    n_show = min(8, len(TRAIN_DEG_PATCHES))
    fig, axes = plt.subplots(2, n_show, figsize=(2 * n_show, 4))
    for i in range(n_show):
        dp = cv2.imread(str(TRAIN_DEG_PATCHES[i]), cv2.IMREAD_GRAYSCALE)
        cp = cv2.imread(str(TRAIN_CLN_PATCHES[i]), cv2.IMREAD_GRAYSCALE)
        axes[0, i].imshow(dp, cmap='gray'); axes[0, i].axis('off')
        axes[1, i].imshow(cp, cmap='gray'); axes[1, i].axis('off')
    axes[0, 0].set_ylabel("Degraded", fontsize=9)
    axes[1, 0].set_ylabel("Clean",    fontsize=9)
    plt.suptitle(f"Sample 128×128 Training Patches  (top: degraded, bottom: clean)", fontsize=11)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"⚠️  Preview skipped: {e}")


## Cell 5 · Classical Enhancement Methods

> These methods require **zero training** and always produce results.
> They serve as a guaranteed baseline and instant quality check.
>
> Methods: **CLAHE · Non-local Means Denoising · Unsharp Masking · Adaptive Thresholding · Combined Pipeline**


In [ ]:
import cv2
import numpy as np

def compute_psnr(img1, img2):
    """PSNR between two uint8 grayscale images."""
    mse = np.mean((img1.astype(np.float32) - img2.astype(np.float32)) ** 2)
    if mse < 1e-10:
        return 100.0  # near-perfect quality; finite value for safe aggregation
    return 10 * np.log10(255 ** 2 / mse)

def compute_ssim(img1, img2):
    """Simplified SSIM between two uint8 grayscale images."""
    I1 = img1.astype(np.float64)
    I2 = img2.astype(np.float64)
    C1, C2 = (0.01 * 255) ** 2, (0.03 * 255) ** 2
    mu1, mu2 = cv2.GaussianBlur(I1, (11,11), 1.5), cv2.GaussianBlur(I2, (11,11), 1.5)
    mu1_sq, mu2_sq, mu1_mu2 = mu1**2, mu2**2, mu1*mu2
    sigma1_sq = cv2.GaussianBlur(I1*I1, (11,11), 1.5) - mu1_sq
    sigma2_sq = cv2.GaussianBlur(I2*I2, (11,11), 1.5) - mu2_sq
    sigma12   = cv2.GaussianBlur(I1*I2, (11,11), 1.5) - mu1_mu2
    num = (2*mu1_mu2 + C1) * (2*sigma12 + C2)
    den = (mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2)
    return float(np.mean(num / (den + 1e-10)))

# ── Classical Methods ─────────────────────────────────────────────────────────
def apply_clahe(img):
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return clahe.apply(img)

def apply_nlm_denoise(img):
    return cv2.fastNlMeansDenoising(img, None, h=10, templateWindowSize=7, searchWindowSize=21)

def apply_unsharp_mask(img, strength=1.5):
    blurred = cv2.GaussianBlur(img, (0, 0), 3)
    sharpened = cv2.addWeighted(img, 1 + strength, blurred, -strength, 0)
    return np.clip(sharpened, 0, 255).astype(np.uint8)

def apply_adaptive_threshold(img):
    return cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                  cv2.THRESH_BINARY, 11, 2)

def apply_combined_pipeline(img):
    """Denoise → Sharpen → CLAHE → Threshold."""
    out = apply_nlm_denoise(img)
    out = apply_unsharp_mask(out, strength=1.0)
    out = apply_clahe(out)
    return out   # skip hard threshold so result stays grayscale for fair metric comparison

print("✅ Classical enhancement functions defined")


In [ ]:
# ── Apply to validation images & compute metrics ──────────────────────────────
try:
    classical_methods = {
        'CLAHE':              apply_clahe,
        'NLM Denoise':        apply_nlm_denoise,
        'Unsharp Mask':       apply_unsharp_mask,
        'Adaptive Threshold': apply_adaptive_threshold,
        'Combined Pipeline':  apply_combined_pipeline,
    }

    n_eval = min(5, len(VAL_DEG))
    classical_psnr = {k: [] for k in classical_methods}
    classical_ssim = {k: [] for k in classical_methods}

    for deg_path, cln_path in zip(VAL_DEG[:n_eval], VAL_CLN[:n_eval]):
        deg = cv2.imread(str(deg_path), cv2.IMREAD_GRAYSCALE)
        cln = cv2.imread(str(cln_path), cv2.IMREAD_GRAYSCALE)
        if cln.shape != deg.shape:
            cln = cv2.resize(cln, (deg.shape[1], deg.shape[0]))
        for name, fn in classical_methods.items():
            enhanced = fn(deg)
            # Adaptive threshold returns binary — resize clean for fair comparison
            if enhanced.shape != cln.shape:
                enhanced = cv2.resize(enhanced, (cln.shape[1], cln.shape[0]))
            classical_psnr[name].append(compute_psnr(enhanced, cln))
            classical_ssim[name].append(compute_ssim(enhanced, cln))

    print("\n📊 Classical Methods — Average Metrics (first 5 validation images)")
    print(f"{'Method':<22} {'PSNR (dB)':>10} {'SSIM':>8}")
    print("-" * 42)
    for name in classical_methods:
        p = np.mean(classical_psnr[name])
        s = np.mean(classical_ssim[name])
        print(f"{name:<22} {p:>10.2f} {s:>8.4f}")

except Exception as e:
    print(f"❌ Classical evaluation error: {e}")
    print("💡 Fix: Check that your validation images are valid grayscale images")


In [ ]:
# ── Visual comparison for one image ───────────────────────────────────────────
try:
    deg = cv2.imread(str(VAL_DEG[0]), cv2.IMREAD_GRAYSCALE)
    cln = cv2.imread(str(VAL_CLN[0]), cv2.IMREAD_GRAYSCALE)
    if cln.shape != deg.shape:
        cln = cv2.resize(cln, (deg.shape[1], deg.shape[0]))

    results = {'Degraded (input)': deg, 'Ground Truth': cln}
    for name, fn in classical_methods.items():
        results[name] = fn(deg)

    n_cols = len(results)
    fig, axes = plt.subplots(1, n_cols, figsize=(3 * n_cols, 4))
    for ax, (title, img) in zip(axes, results.items()):
        ax.imshow(img, cmap='gray')
        ax.set_title(title, fontsize=8)
        ax.axis('off')
    plt.suptitle("Classical Enhancement Comparison", fontsize=12)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"⚠️  Visual comparison skipped: {e}")


## Cell 6 · Data Generator

> Loads patches **one batch at a time** from disk — never loads the entire dataset into RAM.
> Augmentation is applied **identically** to both degraded and clean patches.


In [ ]:
import tensorflow as tf
import numpy as np
import cv2
import random

class DocumentPatchGenerator(tf.keras.utils.Sequence):
    """
    Memory-safe data generator for document image patches.

    - Loads BATCH_SIZE patches per call (never loads whole dataset).
    - Applies identical augmentation to degraded & clean patches.
    - Compatible with mixed precision (returns float32; Keras casts internally).
    """

    def __init__(self, degraded_paths, clean_paths, batch_size,
                 patch_size=128, augment=True, shuffle=True):
        self.deg_paths  = list(degraded_paths)
        self.cln_paths  = list(clean_paths)
        self.batch_size = batch_size
        self.patch_size = patch_size
        self.augment    = augment
        self.shuffle    = shuffle
        self.indices    = list(range(len(self.deg_paths)))
        if self.shuffle:
            random.shuffle(self.indices)

    def __len__(self):
        return max(1, len(self.indices) // self.batch_size)

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        xs, ys = [], []

        for j in batch_idx:
            deg = cv2.imread(str(self.deg_paths[j]), cv2.IMREAD_GRAYSCALE)
            cln = cv2.imread(str(self.cln_paths[j]), cv2.IMREAD_GRAYSCALE)
            if deg is None or cln is None:
                continue  # skip unreadable files; don't add zero patches
            deg = deg.astype(np.float32) / 255.0
            cln = cln.astype(np.float32) / 255.0

            if self.augment:
                deg, cln = self._augment(deg, cln)

            xs.append(deg[:, :, np.newaxis])
            ys.append(cln[:, :, np.newaxis])

        if len(xs) == 0:
            # All files in this batch were unreadable; return empty-ish batch
            s = self.patch_size
            return np.zeros((1, s, s, 1), dtype=np.float32), np.zeros((1, s, s, 1), dtype=np.float32)

        X = np.stack(xs).astype(np.float32)
        Y = np.stack(ys).astype(np.float32)
        return X, Y

    def _augment(self, deg, cln):
        """Apply identical random augmentation to a pair of patches."""
        # Random horizontal flip
        if random.random() < 0.5:
            deg = np.fliplr(deg)
            cln = np.fliplr(cln)
        # Random vertical flip
        if random.random() < 0.5:
            deg = np.flipud(deg)
            cln = np.flipud(cln)
        # Random 90° rotation
        if random.random() < 0.25:
            k = random.choice([1, 2, 3])
            deg = np.rot90(deg, k)
            cln = np.rot90(cln, k)
        # Random brightness ±10%%
        if random.random() < 0.5:
            delta = random.uniform(-0.1, 0.1)
            deg = np.clip(deg + delta, 0.0, 1.0)
            # Apply same brightness shift to clean (preserves paired relationship)
            cln = np.clip(cln + delta, 0.0, 1.0)
        return deg, cln

    def on_epoch_end(self):
        if self.shuffle:
            random.shuffle(self.indices)


print("✅ DocumentPatchGenerator defined")

# ── Instantiate generators ─────────────────────────────────────────────────────
try:
    train_gen = DocumentPatchGenerator(TRAIN_DEG_PATCHES, TRAIN_CLN_PATCHES,
                                       batch_size=BATCH_SIZE, patch_size=PATCH_SIZE,
                                       augment=True, shuffle=True)
    val_gen   = DocumentPatchGenerator(VAL_DEG_PATCHES, VAL_CLN_PATCHES,
                                       batch_size=BATCH_SIZE, patch_size=PATCH_SIZE,
                                       augment=False, shuffle=False)

    print(f"   Train batches : {len(train_gen):,}  ({len(TRAIN_DEG_PATCHES):,} patches)")
    print(f"   Val   batches : {len(val_gen):,}  ({len(VAL_DEG_PATCHES):,} patches)")

    # Quick sanity check
    xb, yb = train_gen[0]
    print(f"   Batch shape   : X={xb.shape}  Y={yb.shape}  dtype={xb.dtype}")
    print(f"   Value range   : [{xb.min():.3f}, {xb.max():.3f}]")
except Exception as e:
    print(f"❌ Generator error: {e}")
    print("💡 Fix: Make sure patch extraction (Cell 4) ran successfully")
    raise


## Cell 7 · Lightweight U-Net Definition

Architecture: 3-level encoder · bottleneck · 3-level decoder · skip connections
- ~500K–1M parameters (right-sized for 300 images)
- Dropout(0.2) + L2(1e-4) regularization to prevent overfitting on small datasets
- Output dtype cast to float32 for mixed-precision compatibility


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, regularizers

def conv_block(x, filters, dropout_rate=0.0, l2=1e-4):
    """Two Conv2D + BatchNorm + ReLU, optional Dropout."""
    reg = regularizers.l2(l2)
    x = layers.Conv2D(filters, 3, padding='same', kernel_regularizer=reg)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, 3, padding='same', kernel_regularizer=reg)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    if dropout_rate > 0:
        x = layers.Dropout(dropout_rate)(x)
    return x

def build_unet(patch_size=128, channels=1, l2=1e-4):
    """Lightweight 3-level U-Net for document enhancement."""
    inputs = layers.Input(shape=(patch_size, patch_size, channels))

    # ── Encoder ────────────────────────────────────────────────────────────
    e1 = conv_block(inputs, 32,  l2=l2)
    p1 = layers.MaxPooling2D(2)(e1)

    e2 = conv_block(p1,  64,  l2=l2)
    p2 = layers.MaxPooling2D(2)(e2)

    e3 = conv_block(p2, 128, l2=l2)
    p3 = layers.MaxPooling2D(2)(e3)

    # ── Bottleneck ─────────────────────────────────────────────────────────
    b  = conv_block(p3, 256, dropout_rate=0.2, l2=l2)

    # ── Decoder ────────────────────────────────────────────────────────────
    d3 = layers.UpSampling2D(2)(b)
    d3 = layers.Concatenate()([d3, e3])
    d3 = conv_block(d3, 128, dropout_rate=0.2, l2=l2)

    d2 = layers.UpSampling2D(2)(d3)
    d2 = layers.Concatenate()([d2, e2])
    d2 = conv_block(d2, 64, dropout_rate=0.2, l2=l2)

    d1 = layers.UpSampling2D(2)(d2)
    d1 = layers.Concatenate()([d1, e1])
    d1 = conv_block(d1, 32, l2=l2)

    # ── Output — cast to float32 for mixed precision ────────────────────────
    logits  = layers.Conv2D(channels, 1, padding='same')(d1)
    outputs = layers.Activation('sigmoid', dtype='float32')(logits)

    return tf.keras.Model(inputs, outputs, name='DocumentUNet')

try:
    model = build_unet(PATCH_SIZE, IMG_CHANNELS)
    model.summary(line_length=80)
    total_params = model.count_params()
    print(f"\n✅ U-Net built — Total parameters: {total_params:,}")
    if total_params < 200_000:
        print("⚠️  Model seems small — double-check architecture")
    elif total_params > 5_000_000:
        print("⚠️  Model seems large for 300 images — consider reducing filters")
    else:
        print("✅ Parameter count looks healthy for ~300 training images")
except Exception as e:
    print(f"❌ Model build error: {e}")
    print("💡 Fix: Make sure TensorFlow is installed and imported correctly")
    raise


## Cell 8 · Loss Function & Metrics

Custom loss = 0.5 × L1 + 0.3 × SSIM + 0.2 × Edge (Sobel) — optimized for sharp text.

In [ ]:
import tensorflow as tf

def document_loss(y_true, y_pred):
    """
    Text-optimized composite loss:
      0.5 × L1   — sharp results (vs blurry MSE)
      0.3 × SSIM — structural similarity (preserves layout)
      0.2 × Edge — Sobel edge loss (preserves text edges)
    """
    # Cast to float32 (safe for mixed precision)
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    # L1 loss
    l1 = tf.reduce_mean(tf.abs(y_true - y_pred))

    # SSIM loss
    ssim = 1.0 - tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=1.0))

    # Edge (Sobel) loss
    true_edges = tf.image.sobel_edges(y_true)
    pred_edges = tf.image.sobel_edges(y_pred)
    edge_loss  = tf.reduce_mean(tf.abs(true_edges - pred_edges))

    return 0.5 * l1 + 0.3 * ssim + 0.2 * edge_loss


def psnr_metric(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    return tf.image.psnr(y_true, y_pred, max_val=1.0)


def ssim_metric(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    return tf.image.ssim(y_true, y_pred, max_val=1.0)


try:
    optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
    model.compile(
        optimizer  = optimizer,
        loss       = document_loss,
        metrics    = [psnr_metric, ssim_metric],
    )
    print("✅ Model compiled")
    print(f"   Optimizer : Adam(lr={LEARNING_RATE})")
    print( "   Loss      : 0.5×L1 + 0.3×SSIM + 0.2×Edge")
    print( "   Metrics   : PSNR, SSIM")
except Exception as e:
    print(f"❌ Compilation error: {e}")
    print("💡 Fix: Make sure Cell 7 (model definition) ran successfully")
    raise


## Cell 9 · Training

> **Expected time:** ~30–45 minutes for 150 epochs on a T4 GPU.
>
> - The best model is saved to Google Drive automatically.
> - If Colab disconnects, re-run all cells — training resumes from the checkpoint.
> - EarlyStopping (patience=30) will stop early if validation loss stops improving.


In [ ]:
import os
from pathlib import Path
import time, gc
import tensorflow as tf

# ── Callbacks ─────────────────────────────────────────────────────────────────
Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath         = CHECKPOINT_PATH,
        monitor          = 'val_loss',
        save_best_only   = True,
        save_weights_only= False,
        verbose          = 1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor  = 'val_loss',
        factor   = 0.5,
        patience = 10,
        min_lr   = 1e-6,
        verbose  = 1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor              = 'val_loss',
        patience             = 30,
        restore_best_weights = True,
        verbose              = 1,
    ),
    tf.keras.callbacks.TensorBoard(
        log_dir       = CHECKPOINT_DIR + 'logs/',
        histogram_freq= 0,
        write_graph   = False,
    ),
]

# ── Resume from checkpoint if available ───────────────────────────────────────
initial_epoch = 0
if Path(CHECKPOINT_PATH).exists():
    try:
        model = tf.keras.models.load_model(
            CHECKPOINT_PATH,
            custom_objects={
                'document_loss': document_loss,
                'psnr_metric'  : psnr_metric,
                'ssim_metric'  : ssim_metric,
            }
        )
        print(f"✅ Resumed from checkpoint: {CHECKPOINT_PATH}")
        # Try to infer initial_epoch from training history saved in logs
        log_dir = Path(CHECKPOINT_DIR + 'logs/')
        if log_dir.exists():
            event_files = list(log_dir.glob('events.out.tfevents.*'))
            if event_files:
                print("   (TensorBoard logs found — continuing training)")
    except Exception as e:
        print(f"⚠️  Could not load checkpoint ({e}) — starting from scratch")
else:
    print("ℹ️  No checkpoint found — starting fresh training")

# ── Train ─────────────────────────────────────────────────────────────────────
try:
    print(f"\n🚀 Starting training  ({EPOCHS} epochs max, EarlyStopping patience=30)")
    start = time.time()

    history = model.fit(
        train_gen,
        validation_data = val_gen,
        epochs          = EPOCHS,
        initial_epoch   = initial_epoch,
        callbacks       = callbacks,
        verbose         = 1,
    )

    elapsed = time.time() - start
    print(f"\n✅ Training complete in {elapsed/60:.1f} minutes")
    print(f"   Best val_loss : {min(history.history['val_loss']):.4f}")
    if 'val_psnr_metric' in history.history:
        print(f"   Best val_PSNR : {max(history.history['val_psnr_metric']):.2f} dB")
    gc.collect()

except Exception as e:
    print(f"❌ Training error: {e}")
    print("💡 Fixes:")
    print("   - OOM error → reduce BATCH_SIZE to 4 in Cell 2, then re-run from Cell 6")
    print("   - Generator error → re-run Cell 4 (patch extraction)")
    print("   - Model error → re-run Cell 7 (model definition)")
    raise


## Cell 10 · Training History Visualization

In [ ]:
try:
    h = history.history
    epochs_ran = range(1, len(h['loss']) + 1)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Loss
    axes[0].plot(epochs_ran, h['loss'],     label='Train Loss')
    axes[0].plot(epochs_ran, h['val_loss'], label='Val Loss')
    axes[0].set_title('Loss (document_loss)')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].legend(); axes[0].grid(True)

    # PSNR
    psnr_key = [k for k in h if 'psnr' in k.lower() and 'val' not in k]
    val_psnr_key = [k for k in h if 'psnr' in k.lower() and 'val' in k]
    if psnr_key:
        axes[1].plot(epochs_ran, h[psnr_key[0]],     label='Train PSNR')
    if val_psnr_key:
        axes[1].plot(epochs_ran, h[val_psnr_key[0]], label='Val PSNR')
    axes[1].set_title('PSNR (dB)'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('PSNR')
    axes[1].legend(); axes[1].grid(True)

    # SSIM
    ssim_key = [k for k in h if 'ssim' in k.lower() and 'val' not in k]
    val_ssim_key = [k for k in h if 'ssim' in k.lower() and 'val' in k]
    if ssim_key:
        axes[2].plot(epochs_ran, h[ssim_key[0]],     label='Train SSIM')
    if val_ssim_key:
        axes[2].plot(epochs_ran, h[val_ssim_key[0]], label='Val SSIM')
    axes[2].set_title('SSIM'); axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('SSIM')
    axes[2].legend(); axes[2].grid(True)

    plt.suptitle('Training History', fontsize=14)
    plt.tight_layout()
    plt.show()

    # Learning-rate curve if available
    if 'lr' in h:
        plt.figure(figsize=(8, 3))
        plt.plot(epochs_ran, h['lr'])
        plt.title('Learning Rate Schedule'); plt.xlabel('Epoch'); plt.ylabel('LR')
        plt.yscale('log'); plt.grid(True)
        plt.tight_layout(); plt.show()

except Exception as e:
    print(f"⚠️  Could not plot history: {e}")
    print("💡 This is non-critical — check the training output above for metrics")


## Cell 11 · Inference & Results

> **Minimum expected values** (after 100–150 epochs on T4):
> - PSNR ≥ 25 dB (good enhancement)
> - SSIM ≥ 0.80 (structural similarity preserved)
>
> If PSNR < 20 dB after training, try: more epochs, lower learning rate, or check data quality.


In [ ]:
import cv2, numpy as np, gc
from pathlib import Path
import matplotlib.pyplot as plt

def enhance_image_unet(model, img_gray_uint8, patch_size=128, stride=64):
    """
    Tile-based inference: splits a full image into overlapping patches,
    runs the model, and blends results back into a full-size output.
    """
    h, w = img_gray_uint8.shape
    img  = img_gray_uint8.astype(np.float32) / 255.0
    out  = np.zeros((h, w), dtype=np.float64)
    cnt  = np.zeros((h, w), dtype=np.float64)

    # Ensure at least one pass over the whole image even if image < patch_size
    y_starts = list(range(0, max(h - patch_size, 0) + 1, stride)) or [0]
    x_starts = list(range(0, max(w - patch_size, 0) + 1, stride)) or [0]

    for y in y_starts:
        for x in x_starts:
            y_end = min(y + patch_size, h)
            x_end = min(x + patch_size, w)
            patch = img[y:y_end, x:x_end]
            ph, pw = patch.shape
            # Pad to patch_size so model always receives correct input shape
            if ph < patch_size or pw < patch_size:
                padded = np.zeros((patch_size, patch_size), dtype=np.float32)
                padded[:ph, :pw] = patch
                patch = padded
            inp  = patch[np.newaxis, :, :, np.newaxis].astype(np.float32)
            pred = model.predict(inp, verbose=0)[0, :ph, :pw, 0]
            out[y:y_end, x:x_end] += pred
            cnt[y:y_end, x:x_end] += 1.0

    # Avoid division by zero
    cnt = np.where(cnt == 0, 1, cnt)
    out = np.clip(out / cnt, 0.0, 1.0)
    return (out * 255).astype(np.uint8)


try:
    # Load best checkpoint
    if Path(CHECKPOINT_PATH).exists():
        best_model = tf.keras.models.load_model(
            CHECKPOINT_PATH,
            custom_objects={
                'document_loss': document_loss,
                'psnr_metric'  : psnr_metric,
                'ssim_metric'  : ssim_metric,
            }
        )
        print(f"✅ Best model loaded from {CHECKPOINT_PATH}")
    else:
        best_model = model
        print("⚠️  Checkpoint not found — using current model weights")

    # Evaluate on up to 10 validation images
    n_show = min(10, len(VAL_DEG))
    psnr_dl_list = []
    ssim_dl_list = []

    Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

    fig, axes = plt.subplots(n_show, 3, figsize=(12, 4 * n_show))
    if n_show == 1:
        axes = axes[np.newaxis, :]

    for i, (deg_path, cln_path) in enumerate(zip(VAL_DEG[:n_show], VAL_CLN[:n_show])):
        deg = cv2.imread(str(deg_path), cv2.IMREAD_GRAYSCALE)
        cln = cv2.imread(str(cln_path), cv2.IMREAD_GRAYSCALE)
        if cln.shape != deg.shape:
            cln = cv2.resize(cln, (deg.shape[1], deg.shape[0]))

        enhanced = enhance_image_unet(best_model, deg, PATCH_SIZE, PATCH_STRIDE)

        p = compute_psnr(enhanced, cln)
        s = compute_ssim(enhanced, cln)
        psnr_dl_list.append(p)
        ssim_dl_list.append(s)

        # Save to Drive
        out_name = f"enhanced_{deg_path.name}"
        cv2.imwrite(str(Path(RESULTS_DIR) / out_name), enhanced)

        axes[i, 0].imshow(deg,      cmap='gray'); axes[i, 0].set_title(f"Degraded",         fontsize=9)
        axes[i, 1].imshow(enhanced, cmap='gray'); axes[i, 1].set_title(f"Enhanced (U-Net)  PSNR={p:.1f}dB  SSIM={s:.3f}", fontsize=9)
        axes[i, 2].imshow(cln,      cmap='gray'); axes[i, 2].set_title(f"Ground Truth",      fontsize=9)
        for ax in axes[i]: ax.axis('off')

    plt.suptitle("Deep Learning Enhancement Results (Degraded | U-Net | Ground Truth)", fontsize=13)
    plt.tight_layout()
    plt.show()

    avg_psnr = np.mean(psnr_dl_list)
    avg_ssim = np.mean(ssim_dl_list)
    print(f"\n📊 Deep Learning Results")
    print(f"   Average PSNR : {avg_psnr:.2f} dB")
    print(f"   Average SSIM : {avg_ssim:.4f}")
    print(f"   Enhanced images saved to: {RESULTS_DIR}")

    if avg_psnr < 20:
        print("⚠️  PSNR < 20 dB — consider: more epochs, check data quality, lower LR")
    elif avg_psnr < 25:
        print("✅ PSNR in acceptable range (20–25 dB)")
    else:
        print("🎉 Excellent PSNR (≥ 25 dB)!")

    gc.collect()

except Exception as e:
    print(f"❌ Inference error: {e}")
    print("💡 Fix: Make sure training (Cell 9) completed and checkpoint was saved")
    raise


## Cell 12 · Method Comparison

Compare classical methods vs deep learning on validation images.

In [ ]:
try:
    # Re-compute classical metrics on same validation images as DL (n_show images)
    n_eval = min(n_show, len(VAL_DEG))
    method_psnr = {}
    method_ssim = {}

    for name, fn in classical_methods.items():
        ps, ss = [], []
        for deg_path, cln_path in zip(VAL_DEG[:n_eval], VAL_CLN[:n_eval]):
            deg = cv2.imread(str(deg_path), cv2.IMREAD_GRAYSCALE)
            cln = cv2.imread(str(cln_path), cv2.IMREAD_GRAYSCALE)
            if cln.shape != deg.shape:
                cln = cv2.resize(cln, (deg.shape[1], deg.shape[0]))
            enh = fn(deg)
            ps.append(compute_psnr(enh, cln))
            ss.append(compute_ssim(enh, cln))
        method_psnr[name] = np.mean(ps)
        method_ssim[name] = np.mean(ss)

    # Add DL result
    method_psnr['U-Net (DL)'] = avg_psnr
    method_ssim['U-Net (DL)'] = avg_ssim

    # Bar chart
    names  = list(method_psnr.keys())
    psnrs  = [method_psnr[n] for n in names]
    ssims  = [method_ssim[n] for n in names]
    colors = ['#4C72B0'] * (len(names) - 1) + ['#DD8452']

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    bars1 = ax1.bar(names, psnrs, color=colors)
    ax1.set_title('PSNR Comparison (dB) — higher is better')
    ax1.set_ylabel('PSNR (dB)')
    ax1.set_xticks(range(len(names)))
    ax1.set_xticklabels(names, rotation=25, ha='right')
    ax1.bar_label(bars1, fmt='%.1f', padding=2)
    ax1.grid(axis='y', alpha=0.4)

    bars2 = ax2.bar(names, ssims, color=colors)
    ax2.set_title('SSIM Comparison — higher is better')
    ax2.set_ylabel('SSIM')
    ax2.set_xticks(range(len(names)))
    ax2.set_xticklabels(names, rotation=25, ha='right')
    ax2.bar_label(bars2, fmt='%.3f', padding=2)
    ax2.grid(axis='y', alpha=0.4)

    plt.suptitle('All Methods Comparison', fontsize=13)
    plt.tight_layout()
    plt.show()

    # Text table
    print(f"\n{'Method':<25} {'PSNR (dB)':>10} {'SSIM':>8}")
    print("-" * 45)
    for name in names:
        marker = ' ← winner' if method_psnr[name] == max(psnrs) else ''
        print(f"{name:<25} {method_psnr[name]:>10.2f} {method_ssim[name]:>8.4f}{marker}")

except Exception as e:
    print(f"⚠️  Comparison plot skipped: {e}")


## Cell 13 · Enhance Your Own Image

Change `test_image_path` to the path of any document image on your Drive.

In [ ]:
# ===== ENHANCE YOUR OWN IMAGE =====
test_image_path = '/content/drive/MyDrive/your_document.png'  # ← change this
# ===================================

def enhance_single_document(image_path, model, patch_size=128, stride=64,
                             save_dir=None):
    """
    Load a single image, enhance it with the U-Net model,
    display the result, and optionally save to Drive.
    """
    try:
        img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise FileNotFoundError(f"Image not found or unreadable: {image_path}")

        enhanced = enhance_image_unet(model, img, patch_size, stride)

        # Display
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        axes[0].imshow(img,      cmap='gray'); axes[0].set_title("Input (Degraded)"); axes[0].axis('off')
        axes[1].imshow(enhanced, cmap='gray'); axes[1].set_title("Enhanced (U-Net)"); axes[1].axis('off')
        plt.suptitle(f"Single Image Enhancement: {Path(image_path).name}", fontsize=12)
        plt.tight_layout(); plt.show()

        # Save
        if save_dir:
            Path(save_dir).mkdir(parents=True, exist_ok=True)
            out_path = str(Path(save_dir) / ('enhanced_' + Path(image_path).name))
        else:
            out_path = str(Path(RESULTS_DIR) / ('enhanced_' + Path(image_path).name))

        cv2.imwrite(out_path, enhanced)
        print(f"✅ Saved to: {out_path}")
        return enhanced

    except FileNotFoundError as e:
        print(f"❌ Error: {e}")
        print(f"💡 Fix: Update 'test_image_path' to a valid image on your Google Drive")
    except Exception as e:
        print(f"❌ Enhancement error: {e}")
        print("💡 Fix: Make sure the model is trained (Cell 9) and the image is valid")


try:
    enhanced_img = enhance_single_document(test_image_path, best_model,
                                            PATCH_SIZE, PATCH_STRIDE, RESULTS_DIR)
except NameError:
    print("⚠️  best_model not found — run Cell 11 first")
except Exception as e:
    print(f"❌ Error: {e}")


## Cell 14 · Export & Cleanup

In [ ]:
import csv, gc
from pathlib import Path

try:
    # ── Save metrics to CSV ────────────────────────────────────────────────
    csv_path = Path(RESULTS_DIR) / 'enhancement_metrics.csv'
    with open(csv_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Method', 'PSNR_dB', 'SSIM'])
        for name in method_psnr:
            writer.writerow([name, f"{method_psnr[name]:.4f}", f"{method_ssim[name]:.4f}"])
    print(f"✅ Metrics CSV saved: {csv_path}")

except NameError:
    print("⚠️  method_psnr not found — run Cells 11 and 12 first")
except Exception as e:
    print(f"⚠️  CSV export skipped: {e}")

try:
    # ── Summary of saved files ─────────────────────────────────────────────
    results_path = Path(RESULTS_DIR)
    saved_files  = list(results_path.iterdir()) if results_path.exists() else []
    print(f"\n📁 Files saved to {RESULTS_DIR}:")
    for f in sorted(saved_files):
        size_kb = f.stat().st_size / 1024
        print(f"   {f.name:<40} {size_kb:>8.1f} KB")

    checkpoint_files = list(Path(CHECKPOINT_DIR).iterdir()) if Path(CHECKPOINT_DIR).exists() else []
    print(f"\n📁 Checkpoint files in {CHECKPOINT_DIR}:")
    for f in sorted(checkpoint_files):
        size_kb = f.stat().st_size / 1024
        print(f"   {f.name:<40} {size_kb:>8.1f} KB")

except Exception as e:
    print(f"⚠️  File listing skipped: {e}")

# ── Memory cleanup ─────────────────────────────────────────────────────────
gc.collect()
tf.keras.backend.clear_session()
print("\n✅ Session complete. All outputs saved to Google Drive.")
print("   Re-run any cell to revisit individual steps.")
